# 校準式空間 GEV 模擬：完整驗證流程

本 notebook 集中執行：

$$
\text{已知空間真值}
\rightarrow
\text{45 annual maxima}
\rightarrow
\text{Frozen NN}
\rightarrow
\text{nested buffered Spatial CV}
\rightarrow
\text{FFS + kernel selection}
\rightarrow
\text{OOF parameter recovery}
\rightarrow
RL_{50},RL_{100}.
$$

模型選擇階段只可使用 NN 估計值與候選 predictors；模擬真值只能在 OOF 預測完成後用於評分。

## 0. 執行設定

第一次只想檢查資料時，執行 generation 與 diagnostics 即可。完整 nested Spatial CV 很耗時，確認前段輸出合理後再開啟。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT if (CURRENT / "src").exists() else CURRENT.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

SIM_DIR = PROJECT_ROOT / "data" / "simulated" / "calibrated_final_model_annual_45"
CV_DIR = SIM_DIR / "nested_spatial_cv_annual"
TIME_PATH = SIM_DIR / "simulation_time.csv"
FIGURE_DIR = Path.home() / "Desktop" / "picture"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = PROJECT_ROOT / "results" / "tables"

RUN_COMPLETE_100_REPLICATES = False  # 確認後改成 True，正式流程可中斷續跑
RUN_GENERATION = False                # 單次 pilot 資料生成
RUN_DIAGNOSTICS = True
RUN_NESTED_SPATIAL_CV = False         # 單次 pilot nested CV
N_REPLICATES = 100
PILOT_REPLICATES = 1
N_YEARS = 45
MONTHS_PER_YEAR = 1
OUTER_FOLDS = 5
INNER_FOLDS = 4
MAX_TRAIN = 800
MIN_TRAIN = 100
MAX_FFS_STEPS = 3
N_JOBS = -2
MAX_ATTEMPTS = 3
RETRY_DELAY_SECONDS = 10
MAX_CONSECUTIVE_FAILURES = 3
DISPLAY_CV_DIR = CV_DIR / "replicate_000"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SIM_DIR =", SIM_DIR)
print("Formal replicates =", N_REPLICATES)

## 正式研究：100 次完整模擬

每個 replicate 都重新生成空間真值與 45 個年最大值，接著執行 Frozen NN、nested buffered Spatial CV、FFS、kernel selection，以及 OOF 參數與 return-level 評估。

100 次依序執行；單一 replicate 內的候選模型使用 `N_JOBS=-2` 平行運算。每次開始與結束都更新 `simulation_time.csv`。

防呆機制不依賴『以前跑過幾次』：程式會實際檢查 annual CSV、欄位、筆數、OOF 輸出與輸入 SHA-256。只有驗證完整才略過；缺檔、損壞或上次中斷的 replicate 會重建，每個 replicate 最多自動重試三次。

In [ ]:
from calibrated_parametric_simulation import CalibratedSimulationConfig
from calibrated_simulation_study import (
    CompleteSimulationStudyConfig,
    run_complete_simulation_study,
)

if RUN_COMPLETE_100_REPLICATES:
    generation_config = CalibratedSimulationConfig(
        n_years=N_YEARS,
        months_per_year=MONTHS_PER_YEAR,
        n_replicates=N_REPLICATES,
        calibration_max_train=MAX_TRAIN,
        n_restarts_optimizer=0,
        seed=20260820,
    )
    study_config = CompleteSimulationStudyConfig(
        n_replicates=N_REPLICATES,
        outer_folds=OUTER_FOLDS,
        inner_folds=INNER_FOLDS,
        max_train=MAX_TRAIN,
        min_train=MIN_TRAIN,
        max_ffs_steps=MAX_FFS_STEPS,
        n_jobs=N_JOBS,
        block_scale="annual",
        save_maxima=True,
        resume=True,
        max_attempts=MAX_ATTEMPTS,
        retry_delay_seconds=RETRY_DELAY_SECONDS,
        max_consecutive_failed_replicates=MAX_CONSECUTIVE_FAILURES,
    )
    study_outputs = run_complete_simulation_study(
        study_config=study_config,
        simulation_config=generation_config,
        output_directory=SIM_DIR,
        cv_root=CV_DIR,
        time_path=TIME_PATH,
    )
else:
    print("尚未啟動正式 100 次；確認設定後將 RUN_COMPLETE_100_REPLICATES 改為 True。")

## RMSE、MAE 與 Bias 的意義

令第 $i$ 個 GRID 的 OOF 誤差為：

$$
e_i=\widehat{y}_i^{\mathrm{OOF}}-y_i^{\mathrm{true}}.
$$

- **RMSE**：$\sqrt{n^{-1}\sum_i e_i^2}$。對大誤差懲罰較重，作為主要準確度指標，越小越好。
- **MAE**：$n^{-1}\sum_i |e_i|$。代表一般 GRID 的平均絕對誤差，越小越好。
- **Bias**：$n^{-1}\sum_i e_i$。正值表示整體高估，負值表示整體低估，越接近 0 越好。

100 次完成後，不只報告單一數值，而是報告各指標跨 replicate 的平均、標準差、中位數與 2.5%--97.5% 模擬區間。

In [ ]:
METRIC_SUMMARY_PATH = SIM_DIR / "simulation_metric_summary.csv"
if TIME_PATH.exists():
    simulation_time = pd.read_csv(TIME_PATH)
    display(simulation_time.tail(10))
    completed_time = simulation_time.loc[simulation_time["status"].eq("completed")]
    if not completed_time.empty:
        last = completed_time.iloc[-1]
        display(pd.DataFrame({
            "completed_replicates": [len(completed_time)],
            "average_time": [last["running_average_hms"]],
            "total_time": [last["cumulative_hms"]],
        }))
else:
    print("尚未產生 simulation_time.csv。")

if METRIC_SUMMARY_PATH.exists():
    metric_summary = pd.read_csv(METRIC_SUMMARY_PATH)
    display(metric_summary)
else:
    print("尚未產生 100 次誤差彙整。")

## 1. 依真實最終模型生成已知真值

生成式使用真實臺灣本島 GRID、真實候選 predictors，以及真實資料最終選定的 mean structure 與 GP kernel。每個 replicate 重新抽出空間隨機效應與 45 個年最大值。

In [ ]:
from calibrated_annual_simulation import (
    annual_config,
    generate_calibrated_annual_replicate,
)
from calibrated_parametric_simulation import prepare_calibrated_simulation

if RUN_GENERATION:
    simulation_config = annual_config(
        n_years=N_YEARS, calibration_max_train=MAX_TRAIN, seed=20260907
    )
    simulation_setup = prepare_calibrated_simulation(
        config=simulation_config, output_directory=SIM_DIR
    )
    generated_paths = generate_calibrated_annual_replicate(
        setup=simulation_setup, config=simulation_config,
        replicate=0, output_directory=SIM_DIR, save_annual_maxima=True,
    )
    display(pd.DataFrame({"generated_file": [str(path) for path in generated_paths.values()]}))
else:
    print("略過重新生成；使用 SIM_DIR 中既有 replicate。")

## 2. 確認 annual maxima、已知參數與 Frozen NN 輸出

每個 GRID 應有 45 年，共 45 個 annual maxima。

In [ ]:
MODEL_READY_PATH = SIM_DIR / "replicate_000_model_ready.csv"
ANNUAL_PATH = SIM_DIR / "replicate_000_annual_maxima.csv"
NN_METRIC_PATH = SIM_DIR / "replicate_000_nn_recovery_metrics.csv"

model_ready = pd.read_csv(MODEL_READY_PATH)
annual_maxima = pd.read_csv(ANNUAL_PATH)
nn_metrics = pd.read_csv(NN_METRIC_PATH)

summary = pd.DataFrame({
    "item": ["GRID count", "annual maxima columns", "xi clipped GRID"],
    "value": [
        len(model_ready),
        sum(column.startswith("annual_max_") for column in annual_maxima.columns),
        int(model_ready["xi_clipped"].sum()),
    ],
})
display(summary)
display(nn_metrics)

## 3. 模擬生成的臺灣年最大溫資料

這裡展示的是由已知空間 GEV 參數實際抽出的 annual maxima，不是參數曲面。三個年份使用共同色階，以比較空間位置與年份間的隨機變動。


In [ ]:
years_to_show = (1980, 2002, 2024)
annual_columns = [f"annual_max_{year}" for year in years_to_show]
plot_data = model_ready[["station", "x_km", "y_km"]].merge(
    annual_maxima[["station", *annual_columns]],
    on="station", how="left", validate="one_to_one",
)
values = plot_data[annual_columns].to_numpy(float)
vmin, vmax = np.nanquantile(values, [0.01, 0.99])
annual_temperature_figure, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for axis, year, column in zip(axes, years_to_show, annual_columns):
    points = axis.scatter(
        plot_data["x_km"], plot_data["y_km"], c=plot_data[column],
        s=10, cmap="magma", vmin=vmin, vmax=vmax,
    )
    axis.set_title(f"Annual maximum: {year}")
    axis.set_aspect("equal")
annual_temperature_figure.colorbar(points, ax=axes, label="Temperature (°C)", shrink=0.78)
annual_temperature_path = FIGURE_DIR / "calibrated_annual_simulation_maxima_examples.png"
annual_temperature_figure.savefig(annual_temperature_path, dpi=220, bbox_inches="tight")
display(annual_temperature_figure)
print("Saved:", annual_temperature_path)


## 4. 檢查模擬曲面是否過度平滑或過度粗糙

比較真實 NN 曲面與模擬真值的共同色階、分布、標準化 variogram 與最近鄰粗糙度。

In [ ]:
from calibrated_simulation_diagnostics import run_diagnostics

if RUN_DIAGNOSTICS:
    diagnostics = run_diagnostics(
        simulation_path=MODEL_READY_PATH,
        figure_directory=FIGURE_DIR,
        table_directory=TABLE_DIR,
    )
    for name in ("surfaces", "distributions", "variograms"):
        display(diagnostics["figures"][name])
    display(diagnostics["roughness"])
else:
    print("略過 diagnostics。")

## 5. 建立 outer geographic folds

Outer folds 只負責最後評估。Outer test fold 與 buffer 內資料不會進入 inner FFS 或 kernel selection。

In [ ]:
from elevation_gp_analysis import prepare_spatial_folds

outer_preview, fold_figure = prepare_spatial_folds(
    model_ready,
    n_folds=OUTER_FOLDS,
    random_state=20260721,
)
fold_figure.savefig(FIGURE_DIR / "calibrated_simulation_outer_folds.png", dpi=220, bbox_inches="tight")
display(fold_figure)
display(
    outer_preview.groupby("spatial_fold")
    .size()
    .rename("n_test_grid")
    .reset_index()
)

## 6. Nested buffered Spatial CV、FFS 與 kernel selection

對每個 outer fold：

1. 保留一區作 outer test。
2. 刪除 test 周圍 target-specific buffer 內的 training GRID。
3. 僅在 outer training 內建立 inner folds。
4. Inner buffered CV 同時執行 grouped FFS 與 RBF/Matérn kernel selection。
5. 用選定模型預測完全未參與選模的 outer test。
6. 五區合併為 OOF predictions。

這一格最耗時。

In [ ]:
from calibrated_annual_simulation import run_annual_evaluation

if RUN_NESTED_SPATIAL_CV:
    cv_outputs = run_annual_evaluation(
        input_path=MODEL_READY_PATH,
        output_directory=DISPLAY_CV_DIR,
        outer_folds=OUTER_FOLDS,
        inner_folds=INNER_FOLDS,
        max_train=MAX_TRAIN,
        min_train=MIN_TRAIN,
        max_steps=MAX_FFS_STEPS,
        min_relative_improvement=0.01,
        maximum_allowed_vif=5.0,
        n_restarts=0,
        n_jobs=N_JOBS,
        random_state=20260721,
    )
else:
    print("尚未重跑 nested Spatial CV；若已有輸出，下一格會直接載入。")

## 7. 載入 nested Spatial CV 結果

In [ ]:
from calibrated_annual_simulation import annual_outputs_current, load_annual_outputs

outputs_are_current = annual_outputs_current(
    MODEL_READY_PATH,
    DISPLAY_CV_DIR,
)

if outputs_are_current:
    cv_outputs = load_annual_outputs(DISPLAY_CV_DIR)
    print("已驗證並載入既有 annual nested Spatial CV 輸出。")
else:
    print("輸出缺漏、損壞或不屬於目前 annual 輸入；將 RUN_NESTED_SPATIAL_CV 改為 True 並重跑第 6 節。")

## 8. 模型選回結果

同一 predictor/kernel 若在多個 outer folds 被選回，表示選模較穩定。只看單一 replicate 不足以估計正式選回率。

In [ ]:
if "cv_outputs" in globals() and "selections" in cv_outputs:
    selections = cv_outputs["selections"]
    display(selections)
    selection_frequency = (
        selections.groupby(
            ["target", "selected_groups", "predictors", "kernel", "nu"],
            dropna=False,
        )
        .size()
        .rename("outer_folds_selected")
        .reset_index()
        .sort_values(["target", "outer_folds_selected"], ascending=[True, False])
    )
    display(selection_frequency)

## 9. OOF 參數恢復

主要比較對象是已知模擬真值，不是 NN reference。

In [ ]:
if "cv_outputs" in globals() and "parameter_metrics" in cv_outputs:
    display(cv_outputs["parameter_metrics"])

    predictions = cv_outputs["predictions"].merge(
        model_ready[["station", "x_km", "y_km"]],
        on="station",
        how="left",
        validate="many_to_one",
    )
    targets = [("mu", r"$\mu$"), ("log_sigma", r"$\log\sigma$"), ("xi", r"$\xi$")]
    columns = [
        ("true_value", "Truth"),
        ("nn_value", "Frozen NN"),
        ("oof_prediction", "Nested OOF GP"),
    ]
    fig, axes = plt.subplots(3, 3, figsize=(12, 14), constrained_layout=True)
    for row, (target, label) in enumerate(targets):
        part = predictions.loc[predictions["target"].eq(target)]
        values = np.concatenate([part[column].to_numpy(float) for column, _ in columns])
        vmin, vmax = np.nanquantile(values, [0.01, 0.99])
        for col, (column, title) in enumerate(columns):
            points = axes[row, col].scatter(
                part["x_km"], part["y_km"], c=part[column],
                s=10, cmap="viridis", vmin=vmin, vmax=vmax,
            )
            axes[row, col].set_title(f"{label}: {title}")
            axes[row, col].set_aspect("equal")
            fig.colorbar(points, ax=axes[row, col], shrink=0.75)
    fig.savefig(FIGURE_DIR / "calibrated_annual_simulation_oof_parameter_recovery.png", dpi=220, bbox_inches="tight")
    display(fig)

## 10. OOF return-level 恢復

$$
\widehat{RL}_{T}^{\mathrm{OOF}}
=
RL_T\!\left(
\widehat\mu^{\mathrm{OOF}},
\widehat{\log\sigma}^{\mathrm{OOF}},
\widehat\xi^{\mathrm{OOF}}
\right).
$$

比較 $\widehat{RL}_{50}^{\mathrm{OOF}}$、$\widehat{RL}_{100}^{\mathrm{OOF}}$ 與生成時已知真值。Annual GEV 直接使用 $p_T=1-1/T$。

In [ ]:
if "cv_outputs" in globals() and "return_level_metrics" in cv_outputs:
    display(cv_outputs["return_level_metrics"])

    rl = cv_outputs["return_level_predictions"].merge(
        model_ready[["station", "x_km", "y_km"]],
        on="station",
        how="left",
        validate="many_to_one",
    )
    fig, axes = plt.subplots(2, 3, figsize=(12, 9), constrained_layout=True)
    columns = [
        ("true_return_level", "Truth"),
        ("nn_return_level", "Frozen NN"),
        ("oof_return_level", "Nested OOF GP"),
    ]
    for row, period in enumerate((50, 100)):
        part = rl.loc[rl["return_period"].eq(period)]
        values = np.concatenate([part[column].to_numpy(float) for column, _ in columns])
        vmin, vmax = np.nanquantile(values, [0.01, 0.99])
        for col, (column, title) in enumerate(columns):
            points = axes[row, col].scatter(
                part["x_km"], part["y_km"], c=part[column],
                s=10, cmap="magma", vmin=vmin, vmax=vmax,
            )
            axes[row, col].set_title(f"RL{period}: {title}")
            axes[row, col].set_aspect("equal")
            fig.colorbar(points, ax=axes[row, col], shrink=0.75)
    fig.savefig(FIGURE_DIR / "calibrated_annual_simulation_oof_return_level_recovery.png", dpi=220, bbox_inches="tight")
    display(fig)

## 11. 結果判讀

完整流程需回答四件事：

- Frozen NN 能否恢復已知 GEV 參數。
- Inner FFS/kernel selection 能否穩定選回生成模型。
- Nested OOF GP 能否恢復未見區域的參數曲面。
- $RL_{50}$ 與 $RL_{100}$ 是否接近已知真值。

單一 replicate 只適合流程測試。正式研究使用 100 個獨立 replicates，報告 RMSE／MAE／Bias 分布、predictor 選回率、kernel 選回率，以及每次與總運算時間。

## 12. 100 次模擬的 RMSE 分布與真實資料比較

RMSE 的重複模擬分布**不應預期為 uniform（均勻分布）**。若流程穩定，RMSE 通常會集中在某個範圍，可能略為右偏；直方圖用來檢查離散程度、偏態與異常 replicate。

下圖使用每次完整流程的 Nested OOF GP RMSE。黑線為 100 次模擬平均，紅線為真實資料 OOF RMSE；表格同時報告標準差、變異係數與 95\% 經驗區間。所有數值只在顯示時四捨五入至小數點後兩位，計算與 CSV 保留原始精度。

> 比較限制：真實資料的參數真值未知，因此真實資料 RMSE 是相對於 NN estimates；模擬資料 RMSE 是相對於已知 simulated truth。兩者可作為校準診斷，但不能解讀成完全相同條件下的優劣檢定。

In [ ]:
from matplotlib.ticker import FormatStrFormatter

SIMULATION_METRICS_PATH = SIM_DIR / "simulation_replicate_metrics.csv"
REAL_PARAMETER_METRICS_PATH = TABLE_DIR / "spatial_ffs_selected_models.csv"
REAL_RL_METRICS_PATH = TABLE_DIR / "spatial_ffs_selected_return_level_metrics.csv"
RMSE_COMPARISON_PATH = TABLE_DIR / "calibrated_annual_simulation_real_vs_simulation_rmse.csv"
RMSE_FIGURE_PATH = FIGURE_DIR / "calibrated_annual_simulation_rmse_histograms.png"

required_paths = [
    SIMULATION_METRICS_PATH,
    REAL_PARAMETER_METRICS_PATH,
    REAL_RL_METRICS_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"缺少 RMSE 輸入檔：{missing_paths}")

outcome_order = ["mu", "log_sigma", "xi", "RL50", "RL100"]
outcome_labels = {
    "mu": r"$\mu$",
    "log_sigma": r"$\log\sigma$",
    "xi": r"$\xi$",
    "RL50": r"$RL_{50}$",
    "RL100": r"$RL_{100}$",
}

simulation_metrics = pd.read_csv(SIMULATION_METRICS_PATH)
simulation_metrics["RMSE"] = pd.to_numeric(simulation_metrics["RMSE"], errors="raise")
sim_rmse = simulation_metrics.loc[
    simulation_metrics["estimator"].eq("Nested OOF GP")
    & simulation_metrics["outcome"].isin(outcome_order),
    ["replicate", "outcome", "RMSE"],
].copy()
if sim_rmse.duplicated(["replicate", "outcome"]).any():
    raise ValueError("同一 replicate/outcome 出現重複的 Nested OOF GP RMSE。")
replicate_counts = sim_rmse.groupby("outcome")["replicate"].nunique().reindex(outcome_order)
if replicate_counts.isna().any() or not replicate_counts.eq(N_REPLICATES).all():
    raise ValueError(
        f"Nested OOF GP RMSE 尚未完整：預期每個 outcome 有 {N_REPLICATES} 次，"
        f"目前為 {replicate_counts.to_dict()}"
    )

real_parameter_rmse = (
    pd.read_csv(REAL_PARAMETER_METRICS_PATH)[["target", "RMSE"]]
    .rename(columns={"target": "outcome", "RMSE": "real_data_oof_rmse"})
)
real_rl_rmse = pd.read_csv(REAL_RL_METRICS_PATH)
real_rl_rmse["outcome"] = "RL" + real_rl_rmse["return_period"].astype(int).astype(str)
real_rl_rmse = real_rl_rmse[["outcome", "RMSE_vs_NN_reference"]].rename(
    columns={"RMSE_vs_NN_reference": "real_data_oof_rmse"}
)
real_rmse = pd.concat([real_parameter_rmse, real_rl_rmse], ignore_index=True)
real_rmse["real_data_oof_rmse"] = pd.to_numeric(
    real_rmse["real_data_oof_rmse"], errors="raise"
)

simulation_summary = (
    sim_rmse.groupby("outcome")["RMSE"]
    .agg(
        n_replicates="size",
        simulation_mean="mean",
        simulation_sd="std",
        simulation_median="median",
        simulation_skewness="skew",
    )
)
simulation_summary["simulation_cv"] = (
    simulation_summary["simulation_sd"] / simulation_summary["simulation_mean"]
)
simulation_summary["simulation_q025"] = sim_rmse.groupby("outcome")["RMSE"].quantile(0.025)
simulation_summary["simulation_q975"] = sim_rmse.groupby("outcome")["RMSE"].quantile(0.975)

rmse_comparison = (
    real_rmse.merge(simulation_summary.reset_index(), on="outcome", how="inner", validate="one_to_one")
    .set_index("outcome")
    .reindex(outcome_order)
    .reset_index()
)
if rmse_comparison["real_data_oof_rmse"].isna().any() or len(rmse_comparison) != len(outcome_order):
    raise ValueError("真實資料與模擬資料的五個 RMSE outcome 無法完整配對。")
rmse_comparison.to_csv(RMSE_COMPARISON_PATH, index=False)

display_table = rmse_comparison[[
    "outcome",
    "real_data_oof_rmse",
    "simulation_mean",
    "simulation_sd",
    "simulation_cv",
    "simulation_median",
    "simulation_q025",
    "simulation_q975",
    "simulation_skewness",
]].rename(columns={
    "outcome": "Outcome",
    "real_data_oof_rmse": "Real-data OOF RMSE",
    "simulation_mean": "Simulation mean",
    "simulation_sd": "Simulation SD",
    "simulation_cv": "Simulation CV",
    "simulation_median": "Simulation median",
    "simulation_q025": "Simulation 2.5%",
    "simulation_q975": "Simulation 97.5%",
    "simulation_skewness": "Skewness",
})
display(display_table.style.format(precision=2))

fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
axes = axes.ravel()
for ax, outcome in zip(axes, outcome_order):
    values = sim_rmse.loc[sim_rmse["outcome"].eq(outcome), "RMSE"].to_numpy(float)
    real_value = float(
        rmse_comparison.loc[rmse_comparison["outcome"].eq(outcome), "real_data_oof_rmse"].iloc[0]
    )
    ax.hist(values, bins=10, color="#4C78A8", edgecolor="white", alpha=0.85)
    ax.axvline(values.mean(), color="black", linewidth=1.8, label="Simulation mean")
    ax.axvline(real_value, color="#D62728", linestyle="--", linewidth=1.8, label="Real-data OOF RMSE")
    ax.set_title(f"{outcome_labels[outcome]}  (n={len(values)})")
    ax.set_xlabel("RMSE")
    ax.set_ylabel("Count")
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.grid(axis="y", alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
axes[-1].axis("off")
axes[-1].legend(handles, labels, loc="center", frameon=False)
axes[-1].text(
    0.5, 0.30,
    "Expected: concentrated, possibly skewed\nNot expected: uniform",
    ha="center", va="center", transform=axes[-1].transAxes,
)
fig.suptitle("Nested OOF GP RMSE across 100 calibrated simulations", fontsize=15)
fig.savefig(RMSE_FIGURE_PATH, dpi=220, bbox_inches="tight")
display(fig)

print("Saved table:", RMSE_COMPARISON_PATH)
print("Saved figure:", RMSE_FIGURE_PATH)

## 13. 同 reference 的 GP-vs-NN RMSE 與假設檢定

為了讓真實資料與模擬資料可比較，兩邊統一計算：

$$
RMSE_{\mathrm{GP-vs-NN}}
=
RMSE\!\left(\widehat\theta_{\mathrm{OOF\ GP}},\widehat\theta_{\mathrm{NN}}\right).
$$

雙尾 Monte Carlo calibration test 的虛無假設為：真實資料 RMSE 與校準模擬分布相容。因同時檢定五個 outcomes，最終判定使用 Holm-adjusted $p$-value；$p_{\mathrm{Holm}}<0.05$ 才判為顯著。原本 GP-vs-truth RMSE 仍保留，用來評估整套流程恢復已知模擬真值的能力。

In [ ]:
SIM_GP_VS_NN_PATH = SIM_DIR / "simulation_gp_vs_nn_rmse.csv"
GP_VS_NN_TEST_PATH = TABLE_DIR / "calibrated_annual_simulation_gp_vs_nn_rmse_test.csv"
GP_VS_NN_BEAMER_TABLE_PATH = TABLE_DIR / "calibrated_annual_monte_carlo_beamer_table.csv"
REAL_PARAMETER_OOF_PATH = TABLE_DIR / "spatial_ffs_selected_oof_predictions.csv"
REAL_RL_OOF_PATH = TABLE_DIR / "spatial_ffs_selected_return_level_oof_predictions.csv"

replicate_directories = sorted(CV_DIR.glob("replicate_*"))
if len(replicate_directories) != N_REPLICATES:
    raise ValueError(
        f"預期 {N_REPLICATES} 個 replicate directories，目前只有 {len(replicate_directories)} 個。"
    )

same_reference_records = []
for replicate_directory in replicate_directories:
    replicate = int(replicate_directory.name.split("_")[-1])
    parameter_path = replicate_directory / "calibrated_annual_nested_predictions.csv"
    return_level_path = replicate_directory / "calibrated_annual_nested_return_level_predictions.csv"
    if not parameter_path.exists() or not return_level_path.exists():
        raise FileNotFoundError(f"replicate {replicate:03d} 缺少 Nested OOF predictions。")

    parameter_predictions = pd.read_csv(
        parameter_path, usecols=["target", "nn_value", "oof_prediction"]
    )
    for outcome, part in parameter_predictions.groupby("target"):
        errors = part["oof_prediction"].to_numpy(float) - part["nn_value"].to_numpy(float)
        same_reference_records.append({
            "replicate": replicate,
            "outcome": outcome,
            "n": len(errors),
            "RMSE_GP_vs_NN": float(np.sqrt(np.mean(errors ** 2))),
        })

    return_level_predictions = pd.read_csv(
        return_level_path,
        usecols=["return_period", "nn_return_level", "oof_return_level"],
    )
    for return_period, part in return_level_predictions.groupby("return_period"):
        errors = part["oof_return_level"].to_numpy(float) - part["nn_return_level"].to_numpy(float)
        same_reference_records.append({
            "replicate": replicate,
            "outcome": f"RL{int(return_period)}",
            "n": len(errors),
            "RMSE_GP_vs_NN": float(np.sqrt(np.mean(errors ** 2))),
        })

simulation_gp_vs_nn = pd.DataFrame(same_reference_records)
if simulation_gp_vs_nn.duplicated(["replicate", "outcome"]).any():
    raise ValueError("同一 replicate/outcome 出現重複的 GP-vs-NN RMSE。")
same_reference_counts = (
    simulation_gp_vs_nn.groupby("outcome")["replicate"].nunique().reindex(outcome_order)
)
if same_reference_counts.isna().any() or not same_reference_counts.eq(N_REPLICATES).all():
    raise ValueError(f"GP-vs-NN RMSE 尚未完整：{same_reference_counts.to_dict()}")
simulation_gp_vs_nn.to_csv(SIM_GP_VS_NN_PATH, index=False)

real_parameter_predictions = pd.read_csv(
    REAL_PARAMETER_OOF_PATH, usecols=["target", "y_true", "y_pred"]
)
real_records = []
for outcome, part in real_parameter_predictions.groupby("target"):
    errors = part["y_pred"].to_numpy(float) - part["y_true"].to_numpy(float)
    real_records.append({
        "outcome": outcome,
        "real_data_rmse_gp_vs_nn": float(np.sqrt(np.mean(errors ** 2))),
    })

real_rl_predictions = pd.read_csv(
    REAL_RL_OOF_PATH, usecols=["return_period", "reference_rl", "predicted_rl"]
)
for return_period, part in real_rl_predictions.groupby("return_period"):
    errors = part["predicted_rl"].to_numpy(float) - part["reference_rl"].to_numpy(float)
    real_records.append({
        "outcome": f"RL{int(return_period)}",
        "real_data_rmse_gp_vs_nn": float(np.sqrt(np.mean(errors ** 2))),
    })
real_gp_vs_nn = pd.DataFrame(real_records)

def two_sided_monte_carlo_pvalue(simulated_values, observed_value):
    simulated_values = np.asarray(simulated_values, dtype=float)
    denominator = len(simulated_values) + 1
    p_lower = (1 + np.sum(simulated_values <= observed_value)) / denominator
    p_upper = (1 + np.sum(simulated_values >= observed_value)) / denominator
    return min(1.0, 2.0 * min(p_lower, p_upper))

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ordered = p_values[order]
    adjusted_ordered = np.maximum.accumulate((len(ordered) - np.arange(len(ordered))) * ordered)
    adjusted = np.empty_like(adjusted_ordered)
    adjusted[order] = np.minimum(adjusted_ordered, 1.0)
    return adjusted

test_records = []
for outcome in outcome_order:
    simulated_values = simulation_gp_vs_nn.loc[
        simulation_gp_vs_nn["outcome"].eq(outcome), "RMSE_GP_vs_NN"
    ].to_numpy(float)
    observed_value = float(
        real_gp_vs_nn.loc[
            real_gp_vs_nn["outcome"].eq(outcome), "real_data_rmse_gp_vs_nn"
        ].iloc[0]
    )
    test_records.append({
        "outcome": outcome,
        "real_data_rmse_gp_vs_nn": observed_value,
        "simulation_mean_rmse_gp_vs_nn": float(np.mean(simulated_values)),
        "simulation_q025_rmse_gp_vs_nn": float(np.quantile(simulated_values, 0.025)),
        "simulation_q975_rmse_gp_vs_nn": float(np.quantile(simulated_values, 0.975)),
        "monte_carlo_p_value": two_sided_monte_carlo_pvalue(simulated_values, observed_value),
    })

gp_vs_nn_test = pd.DataFrame(test_records)
gp_vs_nn_test["holm_adjusted_p_value"] = holm_adjust(
    gp_vs_nn_test["monte_carlo_p_value"].to_numpy(float)
)
gp_vs_nn_test["decision_alpha_0.05"] = np.where(
    gp_vs_nn_test["holm_adjusted_p_value"] < 0.05,
    "Significant",
    "Not significant",
)
gp_vs_nn_test.to_csv(GP_VS_NN_TEST_PATH, index=False)

outcome_display = {"mu": r"$\mu$", "log_sigma": r"$\log\sigma$", "xi": r"$\xi$", "RL50": r"$RL_{50}$", "RL100": r"$RL_{100}$"}
beamer_table = pd.DataFrame({
    "Outcome": gp_vs_nn_test["outcome"].map(outcome_display),
    "Simulation RMSE": [
        f"{mean:.3f} [{lower:.3f}, {upper:.3f}]"
        for mean, lower, upper in zip(
            gp_vs_nn_test["simulation_mean_rmse_gp_vs_nn"],
            gp_vs_nn_test["simulation_q025_rmse_gp_vs_nn"],
            gp_vs_nn_test["simulation_q975_rmse_gp_vs_nn"],
        )
    ],
    "Annual RMSE": gp_vs_nn_test["real_data_rmse_gp_vs_nn"],
    "Hypothesis-test result": [
        f"{decision} (Holm p={p_value:.3f})"
        for decision, p_value in zip(
            gp_vs_nn_test["decision_alpha_0.05"],
            gp_vs_nn_test["holm_adjusted_p_value"],
        )
    ],
})
beamer_table.to_csv(GP_VS_NN_BEAMER_TABLE_PATH, index=False)
display(beamer_table.style.format({"Annual RMSE": "{:.3f}"}).hide(axis="index"))
print("Saved per-replicate GP-vs-NN RMSE:", SIM_GP_VS_NN_PATH)
print("Saved hypothesis-test table:", GP_VS_NN_TEST_PATH)
print("Saved compact Beamer table:", GP_VS_NN_BEAMER_TABLE_PATH)